<a href="https://colab.research.google.com/github/florianaewing/CSB430SWIWinter2026/blob/main/OptimumDeparture_2014_2024V4_FE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Florian Ewing, Margaret Keu, John Farnandez, Lauren Connelly
# Professor Ix Procopios
# Software Design and Implementation
# Adapted: 2014–2024 BTS Flight Delay Dataset

# Optimum Departure Location & Airline by State Predictor
# (2014–2024 BTS Dataset Adaptation)

This notebook builds a machine learning model that predicts whether a U.S. domestic flight
will depart on time or experience a significant delay, then uses that model to recommend
the best airport and airline combination for a traveler to minimize their delay risk.

The dataset is the **2014–2024 Delayed Flight Data** (`JOHNFF09/2014-2024-delayed-flight-data`)
sourced from Kaggle, which contains over 63 million flights exported directly from the
Bureau of Transportation Statistics (BTS) on-time performance database. Because BTS exports
encode every categorical field as a numeric code, three official BTS lookup tables are fetched
at runtime to decode airport IDs to city/state labels, city market IDs to city names, and
carrier codes to full airline names.

The model uses a **binary classification target**:
- **Class 0** → Flight departed on time or fewer than 15 minutes late
- **Class 1** → Flight departed 15 or more minutes late

Features used for prediction include the airline, departure airport, year, month, day of week,
weekend flag, holiday proximity flag, and scheduled departure hour. All features are available
at booking time — no post-departure information is used. A GPU-accelerated XGBoost classifier
is trained on the full 62 million row dataset using 5-fold stratified cross-validation on an
NVIDIA A100 GPU. Class imbalance (~82% on time) is handled via the scale_pos_weight parameter.

The recommender accepts a **U.S. state** and departure date from the user, evaluates every
airport and airline combination that has historically operated in that state, and returns
the top 5 combinations with the lowest predicted delay probability. State-level granularity
ensures that a traveler in Washington is not recommended airports in Colorado.

The trained model and label encoders are saved to disk so the predictor can be loaded and
used in other environments without retraining.

# Cell 1 — Download Dataset

Downloads the 2014–2024 BTS flight delay dataset from Kaggle using `kagglehub`,
which handles authentication automatically via Colab's built-in Kaggle integration.
Also imports all libraries used throughout the notebook.

In [1]:
# ===== Cell 1: Download Dataset =====
import os
import pandas as pd
import numpy as np
import kagglehub

dataset_path = kagglehub.dataset_download("JOHNFF09/2014-2024-delayed-flight-data")
print("Downloaded to:", dataset_path)

print("\nFiles:")
for f in sorted(os.listdir(dataset_path)):
    size_mb = os.path.getsize(os.path.join(dataset_path, f)) / 1e6
    print(f"  {f:50s}  {size_mb:7.1f} MB")

100%|██████████| 975M/975M [00:52<00:00, 19.6MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/JOHNFF09/2014-2024-delayed-flight-data/versions/1

Files:
  flights10year.csv                                    6002.9 MB


# Cell 2 — Fetch BTS Lookup Tables

Fetches three lookup tables directly from the BTS transtats server to decode
the numeric codes stored in the raw dataset into human-readable values.
The airport lookup maps numeric airport IDs to city and state labels.
The city market lookup provides a backup state source for any airports
not resolved by the airport lookup. The carrier lookup maps two-letter
carrier codes to full airline names.

In [2]:
# ===== Cell 2: Fetch BTS Lookup Tables =====
import requests, io, zipfile

BTS_LOOKUP_URLS = {
    "airport":     "https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_NVecbeg_VQ",
    "city_market": "https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_PVgl_ZNeXRg_VQ",
    "carrier":     "https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_haVdhR_PNeeVRef",
}

def fetch_bts_lookup(url):
    """Fetch a BTS lookup table. Handles both plain CSV and zip responses."""
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    raw = resp.content
    if raw[:2] == b"PK":
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            csv_name = [n for n in z.namelist() if n.endswith(".csv")][0]
            with z.open(csv_name) as f:
                return pd.read_csv(f)
    else:
        return pd.read_csv(io.StringIO(raw.decode("utf-8", errors="replace")))

# Airport ID lookup — format: "City, ST: Airport Name"
raw_airport = fetch_bts_lookup(BTS_LOOKUP_URLS["airport"])
raw_airport.columns = ["Code", "Description"]
raw_airport["Code"] = pd.to_numeric(raw_airport["Code"], errors="coerce")
raw_airport = raw_airport.dropna(subset=["Code"])
raw_airport["Code"] = raw_airport["Code"].astype(int)
raw_airport["State"]     = raw_airport["Description"].str.extract(r",\s*([A-Z]{2}):")
raw_airport["CityState"] = raw_airport["Description"].str.extract(r"^(.+):")
airport_lookup = raw_airport[["Code", "State", "CityState"]].dropna(subset=["State"])
print(f"Airport lookup: {len(airport_lookup)} entries")

# City market lookup — format: "City, ST"
raw_city = fetch_bts_lookup(BTS_LOOKUP_URLS["city_market"])
raw_city.columns = ["Code", "Description"]
raw_city["Code"] = pd.to_numeric(raw_city["Code"], errors="coerce")
raw_city = raw_city.dropna(subset=["Code"])
raw_city["Code"] = raw_city["Code"].astype(int)
raw_city["State"]    = raw_city["Description"].str.extract(r",\s*([A-Z]{2})$")
raw_city["CityName"] = raw_city["Description"].str.extract(r"^(.+),\s*[A-Z]{2}$")
city_lookup = raw_city[["Code", "CityName", "State"]].dropna(subset=["State"])
print(f"City market lookup: {len(city_lookup)} entries")

# Carrier lookup
raw_carrier = fetch_bts_lookup(BTS_LOOKUP_URLS["carrier"])
raw_carrier.columns = ["Code", "Description"]
raw_carrier = raw_carrier.dropna()
raw_carrier["Code"]        = raw_carrier["Code"].astype(str).str.strip()
raw_carrier["AirlineName"] = raw_carrier["Description"].str.strip()
carrier_lookup = raw_carrier[["Code", "AirlineName"]]
print(f"Carrier lookup: {len(carrier_lookup)} entries")

Airport lookup: 2961 entries
City market lookup: 2491 entries
Carrier lookup: 1774 entries


# Cell 3 — Load Flight Data and Decode Coded Fields

Loads the raw CSV and renames BTS column names to shorter internal names.
Immediately joins all three lookup tables to add human-readable fields
alongside the numeric IDs. Both forms are retained: numeric IDs are used
by the model for encoding, while the decoded labels are used in output.

In [3]:
# ===== Cell 3: Load Flight Data and Decode =====

KEEP_RAW = [
    "FL_DATE", "OP_UNIQUE_CARRIER",
    "ORIGIN_AIRPORT_ID", "ORIGIN_CITY_MARKET_ID",
    "DEST_AIRPORT_ID",   "DEST_CITY_MARKET_ID",
    "CRS_DEP_TIME", "DEP_TIME", "DEP_DELAY_NEW", "ARR_DELAY_NEW",
    "DISTANCE", "CARRIER_DELAY", "WEATHER_DELAY",
    "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
]

BTS_RENAME = {
    "FL_DATE":               "FlightDate",
    "OP_UNIQUE_CARRIER":     "Airline_Code",
    "ORIGIN_AIRPORT_ID":     "Dep_Airport_ID",
    "ORIGIN_CITY_MARKET_ID": "Dep_Market_ID",
    "DEST_AIRPORT_ID":       "Arr_Airport_ID",
    "DEST_CITY_MARKET_ID":   "Arr_Market_ID",
    "CRS_DEP_TIME":          "Sched_Dep_Time",
    "DEP_TIME":              "Dep_Time",
    "DEP_DELAY_NEW":         "Dep_Delay",
    "ARR_DELAY_NEW":         "Arr_Delay",
    "DISTANCE":              "Distance",
    "CARRIER_DELAY":         "Carrier_Delay",
    "WEATHER_DELAY":         "Weather_Delay",
    "NAS_DELAY":             "NAS_Delay",
    "SECURITY_DELAY":        "Security_Delay",
    "LATE_AIRCRAFT_DELAY":   "LateAircraft_Delay",
}

csv_files = sorted([
    os.path.join(dataset_path, f)
    for f in os.listdir(dataset_path) if f.endswith(".csv")
])

print(f"Found {len(csv_files)} CSV file(s). Loading...")
chunks = []
for fp in csv_files:
    header = pd.read_csv(fp, nrows=0).columns.tolist()
    cols_to_read = [c for c in KEEP_RAW if c in header]
    df = pd.read_csv(fp, usecols=cols_to_read, low_memory=False)
    df.rename(columns=BTS_RENAME, inplace=True)
    df["FlightDate"] = pd.to_datetime(df["FlightDate"], format="mixed")
    chunks.append(df)
    print(f"  Loaded {fp.split('/')[-1]:50s}  rows={len(df):,}")

flights = pd.concat(chunks, ignore_index=True)
print(f"\nTotal rows loaded: {len(flights):,}")

# Decode airport IDs
flights = flights.merge(
    airport_lookup.rename(columns={"Code": "Dep_Airport_ID", "State": "Dep_State", "CityState": "Dep_CityState"}),
    on="Dep_Airport_ID", how="left"
)
flights = flights.merge(
    airport_lookup.rename(columns={"Code": "Arr_Airport_ID", "State": "Arr_State", "CityState": "Arr_CityState"}),
    on="Arr_Airport_ID", how="left"
)

# Decode city market IDs (backup state source)
flights = flights.merge(
    city_lookup.rename(columns={"Code": "Dep_Market_ID", "CityName": "Dep_CityName", "State": "Dep_Market_State"}),
    on="Dep_Market_ID", how="left"
)
flights["Dep_State"] = flights["Dep_State"].fillna(flights["Dep_Market_State"])

# Decode carrier codes
flights = flights.merge(
    carrier_lookup.rename(columns={"Code": "Airline_Code", "AirlineName": "Airline_Name"}),
    on="Airline_Code", how="left"
)

print(f"\nDecoded flights shape: {flights.shape}")
print("\nSample decoded rows:")
print(flights[
    ["FlightDate", "Airline_Code", "Airline_Name",
     "Dep_Airport_ID", "Dep_CityState", "Dep_State", "Dep_Delay"]
].dropna(subset=["Airline_Name", "Dep_CityState"]).head(5).to_string(index=False))

Found 1 CSV file(s). Loading...
  Loaded flights10year.csv                                   rows=63,626,717

Total rows loaded: 63,626,717

Decoded flights shape: (63626717, 23)

Sample decoded rows:
FlightDate Airline_Code      Airline_Name  Dep_Airport_ID                  Dep_CityState Dep_State  Dep_Delay
2024-01-01           9E Endeavor Air Inc.           10135 Allentown/Bethlehem/Easton, PA        PA        0.0
2024-01-01           9E Endeavor Air Inc.           10135 Allentown/Bethlehem/Easton, PA        PA        9.0
2024-01-01           9E Endeavor Air Inc.           10185                 Alexandria, LA        LA        0.0
2024-01-01           9E Endeavor Air Inc.           10185                 Alexandria, LA        LA        0.0
2024-01-01           9E Endeavor Air Inc.           10208                    Augusta, GA        GA        0.0


# Cell 4 — Create Target Variables and Class Weights

Drops rows with no recorded departure delay (cancelled or diverted flights).
Creates the binary classification target: 0 for on-time or minor delays
under 15 minutes, 1 for delays of 15 minutes or more. Computes the
positive class weight ratio used by XGBoost to compensate for the
imbalanced class distribution (~82% on time).

In [4]:
# ===== Cell 4: Target Variables =====

flights = flights.dropna(subset=["Dep_Delay"])
flights["Delay_Class"] = np.where(flights["Dep_Delay"] >= 15, 1, 0)

delay_counts = flights["Delay_Class"].value_counts()
delay_pct    = flights["Delay_Class"].value_counts(normalize=True) * 100

print("Delay Class Counts:");  print(delay_counts)
print("\nDelay Class %:");      print(delay_pct.round(2))

total    = len(flights)
weight_0 = total / (2 * delay_counts[0])
weight_1 = total / (2 * delay_counts[1])
class_weights = {0: weight_0, 1: weight_1}
print("\nClass weights:", class_weights)

Delay Class Counts:
Delay_Class
0    50927248
1    11470970
Name: count, dtype: int64

Delay Class %:
Delay_Class
0    81.62
1    18.38
Name: proportion, dtype: float64

Class weights: {0: np.float64(0.6126211453640692), 1: np.float64(2.719831801495427)}


# Cell 5 — State Mapping

Tags every flight with the U.S. state of its departure airport using the
decoded Dep_State field from Cell 3. Builds a lookup of all valid state
abbreviations present in the dataset and a full state name dictionary
used to display readable names in the recommender output.

In [5]:
# ===== Cell 5: State Mapping =====

STATE_NAMES = {
    "AL": "Alabama",       "AK": "Alaska",        "AZ": "Arizona",
    "AR": "Arkansas",      "CA": "California",    "CO": "Colorado",
    "CT": "Connecticut",   "DE": "Delaware",       "FL": "Florida",
    "GA": "Georgia",       "HI": "Hawaii",         "ID": "Idaho",
    "IL": "Illinois",      "IN": "Indiana",        "IA": "Iowa",
    "KS": "Kansas",        "KY": "Kentucky",       "LA": "Louisiana",
    "ME": "Maine",         "MD": "Maryland",       "MA": "Massachusetts",
    "MI": "Michigan",      "MN": "Minnesota",      "MS": "Mississippi",
    "MO": "Missouri",      "MT": "Montana",        "NE": "Nebraska",
    "NV": "Nevada",        "NH": "New Hampshire",  "NJ": "New Jersey",
    "NM": "New Mexico",    "NY": "New York",       "NC": "North Carolina",
    "ND": "North Dakota",  "OH": "Ohio",           "OK": "Oklahoma",
    "OR": "Oregon",        "PA": "Pennsylvania",   "RI": "Rhode Island",
    "SC": "South Carolina","SD": "South Dakota",   "TN": "Tennessee",
    "TX": "Texas",         "UT": "Utah",           "VT": "Vermont",
    "VA": "Virginia",      "WA": "Washington",     "WV": "West Virginia",
    "WI": "Wisconsin",     "WY": "Wyoming",
}

valid_states = sorted(flights["Dep_State"].dropna().unique().tolist())
valid_states = [s for s in valid_states if s in STATE_NAMES]

print(f"States with flight data: {len(valid_states)}")
print(flights["Dep_State"].value_counts().head(10))
print(f"\nUnmapped (territories etc.): {flights['Dep_State'].isna().sum():,}")

States with flight data: 50
Dep_State
CA    7016657
TX    6674187
FL    5246603
GA    3676173
IL    3609062
NY    3009501
CO    2746458
NC    2516349
AZ    1879505
NV    1752271
Name: count, dtype: int64

Unmapped (territories etc.): 0


# Cell 6 — Feature Engineering

Extracts temporal features from the flight date: year, month, day of week,
weekend flag, and a holiday proximity flag that marks flights within 2 days
of a major U.S. holiday across all years 2014–2024. Adds a departure hour
feature from the scheduled departure time. All features are available at
booking time — no post-departure information is included. Encodes the
airline code and airport ID as integers for the model.

In [6]:
# ===== Cell 6: Feature Engineering =====
from sklearn.preprocessing import LabelEncoder

# Holiday proximity flag (2014–2024)
holiday_dates = []
for year in range(2014, 2025):
    holiday_dates += [f"{year}-01-01", f"{year}-07-04", f"{year}-12-25"]

approx_floating = [
    "2014-01-20","2014-02-17","2014-05-26","2014-09-01","2014-11-27",
    "2015-01-19","2015-02-16","2015-05-25","2015-09-07","2015-11-26",
    "2016-01-18","2016-02-15","2016-05-30","2016-09-05","2016-11-24",
    "2017-01-16","2017-02-20","2017-05-29","2017-09-04","2017-11-23",
    "2018-01-15","2018-02-19","2018-05-28","2018-09-03","2018-11-22",
    "2019-01-21","2019-02-18","2019-05-27","2019-09-02","2019-11-28",
    "2020-01-20","2020-02-17","2020-05-25","2020-09-07","2020-11-26",
    "2021-01-18","2021-02-15","2021-05-31","2021-09-06","2021-11-25",
    "2022-01-17","2022-02-21","2022-05-30","2022-09-05","2022-11-24",
    "2023-01-16","2023-02-20","2023-05-29","2023-09-04","2023-11-23",
    "2024-01-15","2024-02-19","2024-05-27","2024-09-02","2024-11-28",
]
us_holidays = pd.to_datetime(holiday_dates + approx_floating)

def is_near_holiday(date_series, holidays, window=2):
    flags = np.zeros(len(date_series), dtype=int)
    for h in holidays:
        diff = (date_series - h).dt.days.abs()
        flags |= (diff <= window).astype(int).values
    return flags

flights["Year"]      = flights["FlightDate"].dt.year
flights["Month"]     = flights["FlightDate"].dt.month
flights["DayOfWeek"] = flights["FlightDate"].dt.dayofweek
flights["IsWeekend"] = (flights["DayOfWeek"] >= 5).astype(int)
flights["IsHoliday"] = is_near_holiday(flights["FlightDate"], us_holidays)
flights["Dep_Hour"]  = (flights["Sched_Dep_Time"] // 100).clip(0, 23)

FEATURE_COLS = [
    "Airline_Code", "Dep_Airport_ID", "Year", "Month", "DayOfWeek",
    "IsWeekend", "IsHoliday", "Dep_Hour",
]

model_df = flights[
    FEATURE_COLS + ["Delay_Class", "Dep_State", "Airline_Name", "Dep_CityState"]
].dropna(subset=FEATURE_COLS + ["Delay_Class", "Dep_State"])

le_airline = LabelEncoder()
le_airport = LabelEncoder()

model_df = model_df.copy()
model_df["Airline_enc"]     = le_airline.fit_transform(model_df["Airline_Code"])
model_df["Dep_Airport_enc"] = le_airport.fit_transform(model_df["Dep_Airport_ID"])

FINAL_FEATURES = [
    "Airline_enc", "Dep_Airport_enc", "Year", "Month", "DayOfWeek",
    "IsWeekend", "IsHoliday", "Dep_Hour",
]

X = model_df[FINAL_FEATURES]
y = model_df["Delay_Class"]

print(f"Feature matrix shape: {X.shape}")
print(f"Class distribution:\n{y.value_counts(normalize=True).round(3)}")

Feature matrix shape: (62398218, 8)
Class distribution:
Delay_Class
0    0.816
1    0.184
Name: proportion, dtype: float64


# Cell 7 — Train GPU-Accelerated XGBoost with 5-Fold Cross-Validation

Trains an XGBoost gradient boosted tree classifier on the full 62 million row
dataset using the A100 GPU via the device='cuda' and tree_method='hist' parameters.
Class imbalance is handled via scale_pos_weight rather than oversampling.
5-fold stratified cross-validation is run sequentially to produce a stable
generalization estimate with variance. A final model is then trained on the
full dataset and evaluated on a held-out 20% test split.

In [7]:
# ===== Cell 7: Train GPU-Accelerated XGBoost with 5-Fold Cross-Validation =====
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, f1_score
import numpy as np

print(f"XGBoost version: {xgb.__version__}")
print(f"Training on full dataset: {len(model_df):,} rows")

X_s = model_df[FINAL_FEATURES].astype("float32")
y_s = model_df["Delay_Class"].astype("int32")

n0 = (y_s == 0).sum()
n1 = (y_s == 1).sum()
scale_pos_weight = n0 / n1
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

params = dict(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    device="cuda",
    tree_method="hist",
    eval_metric="logloss",
    random_state=42,
)

# 5-fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_s, y_s), 1):
    X_train, X_val = X_s.iloc[train_idx].values, X_s.iloc[val_idx].values
    y_train, y_val = y_s.iloc[train_idx].values, y_s.iloc[val_idx].values

    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    dval  = xgb.DMatrix(X_val)
    y_prob = model.get_booster().predict(dval)
    y_pred = (y_prob > 0.5).astype(int)

    f1 = f1_score(y_val, y_pred, average="weighted")
    fold_scores.append(f1)
    print(f"  Fold {fold} — weighted F1: {f1:.4f}")
    del model

print(f"\nCross-validation weighted F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

# Final model on full dataset
print("\nTraining final model on full dataset...")
rf = xgb.XGBClassifier(**params)
rf.fit(X_s, y_s, verbose=False)

# Evaluate on held-out 20% test split
X_train_f, X_test, y_train_f, y_test = train_test_split(
    X_s, y_s, test_size=0.2, random_state=99, stratify=y_s
)
y_pred_final = rf.predict(X_test)

print("\nFinal model classification report:")
print(classification_report(y_test, y_pred_final, target_names=["On Time", "Delayed"]))

XGBoost version: 3.2.0
Training on full dataset: 62,398,218 rows
scale_pos_weight: 4.4397
  Fold 1 — weighted F1: 0.6912
  Fold 2 — weighted F1: 0.6912
  Fold 3 — weighted F1: 0.6916
  Fold 4 — weighted F1: 0.6913
  Fold 5 — weighted F1: 0.6915

Cross-validation weighted F1: 0.6914 ± 0.0002

Training final model on full dataset...


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [02:21:30] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



Final model classification report:
              precision    recall  f1-score   support

     On Time       0.89      0.65      0.76  10185450
     Delayed       0.30      0.66      0.41   2294194

    accuracy                           0.65  12479644
   macro avg       0.60      0.66      0.58  12479644
weighted avg       0.78      0.65      0.69  12479644



# Cell 8 — Save Model and Encoders

Saves the trained XGBoost model in its native binary format and both label
encoders via joblib. Also writes a JSON metadata file containing the feature
list and valid states so a loader knows exactly what the model expects.
Files are saved to /content/model/ and can be downloaded from the Colab
file browser on the left sidebar.

In [8]:
# ===== Cell 8: Save Model and Encoders =====
import joblib, json

SAVE_DIR = "/content/model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save XGBoost model in native format
rf.save_model(f"{SAVE_DIR}/xgboost_model.json")

# Save label encoders
joblib.dump(le_airline, f"{SAVE_DIR}/le_airline.joblib")
joblib.dump(le_airport, f"{SAVE_DIR}/le_airport.joblib")

# Save metadata
with open(f"{SAVE_DIR}/model_meta.json", "w") as f:
    json.dump({
        "final_features": FINAL_FEATURES,
        "valid_states":   valid_states,
    }, f, indent=2)

print("Saved:")
for fname in os.listdir(SAVE_DIR):
    size_kb = os.path.getsize(os.path.join(SAVE_DIR, fname)) / 1e3
    print(f"  {fname:35s}  {size_kb:8.1f} KB")

print("\nTo load the model elsewhere:")
print("  import xgboost as xgb, joblib")
print("  rf         = xgb.XGBClassifier()")
print("  rf.load_model('xgboost_model.json')")
print("  le_airline = joblib.load('le_airline.joblib')")
print("  le_airport = joblib.load('le_airport.joblib')")

Saved:
  model_meta.json                           0.7 KB
  xgboost_model.json                    14820.2 KB
  le_airport.joblib                         3.4 KB
  le_airline.joblib                         0.6 KB

To load the model elsewhere:
  import xgboost as xgb, joblib
  rf         = xgb.XGBClassifier()
  rf.load_model('xgboost_model.json')
  le_airline = joblib.load('le_airline.joblib')
  le_airport = joblib.load('le_airport.joblib')


# Cell 9 — Recommender Function

Defines the recommender that accepts a U.S. state abbreviation, a departure
date, and an optional departure hour. Evaluates every airport and airline
combination that historically operated in that state, builds a feature row
for each using the user's date and the airport's median historical departure
hour, and predicts the delay probability for each combination. Returns the
top 5 with the lowest delay probability using fully decoded human-readable
airport city labels and airline names.

In [9]:
# ===== Cell 9: Recommender Function =====

def get_top5_recommendations(state, date_str, dep_hour=None):
    """
    Given a U.S. state abbreviation (e.g. 'WA'), a date string (YYYY-MM-DD),
    and an optional departure hour (0-23), return the top 5 airport+airline
    combinations with the lowest predicted delay probability.
    """
    date = pd.to_datetime(date_str)

    year       = date.year
    month      = date.month
    dow        = date.dayofweek
    is_weekend = int(dow >= 5)
    is_holiday = int(any(abs((date - h).days) <= 2 for h in us_holidays))

    # All airport+airline combos that operated in this state
    combos = (
        model_df[model_df["Dep_State"] == state]
        [["Dep_Airport_ID", "Airline_Code", "Airline_Name", "Dep_CityState"]]
        .drop_duplicates(subset=["Dep_Airport_ID", "Airline_Code"])
        .reset_index(drop=True)
    )

    if combos.empty:
        print(f"No flight combinations found for state: {state}")
        return None

    # Median departure hour per airport from historical data
    airport_medians = (
        model_df[model_df["Dep_State"] == state]
        .groupby("Dep_Airport_ID")[["Dep_Hour"]]
        .median()
        .reset_index()
    )
    combos = combos.merge(airport_medians, on="Dep_Airport_ID", how="left")

    if dep_hour is not None:
        combos["Dep_Hour"] = int(dep_hour)

    combos["Year"]      = year
    combos["Month"]     = month
    combos["DayOfWeek"] = dow
    combos["IsWeekend"] = is_weekend
    combos["IsHoliday"] = is_holiday

    def safe_encode(encoder, values):
        known = set(encoder.classes_)
        return np.array([
            encoder.transform([v])[0] if v in known else -1
            for v in values
        ])

    combos["Airline_enc"]     = safe_encode(le_airline, combos["Airline_Code"])
    combos["Dep_Airport_enc"] = safe_encode(le_airport, combos["Dep_Airport_ID"])

    combos = combos[
        (combos["Airline_enc"]     != -1) &
        (combos["Dep_Airport_enc"] != -1)
    ]

    X_pred = combos[FINAL_FEATURES].astype("float32")
    delay_probs = rf.predict_proba(X_pred)[:, 1]

    combos["Delay_Probability"]  = delay_probs
    combos["OnTime_Probability"] = 1 - delay_probs

    top5 = (
        combos[["Dep_CityState", "Airline_Name", "Delay_Probability", "OnTime_Probability"]]
        .sort_values("Delay_Probability")
        .head(5)
        .reset_index(drop=True)
    )
    top5.index += 1
    top5["Delay_Probability"]  = top5["Delay_Probability"].map("{:.1%}".format)
    top5["OnTime_Probability"] = top5["OnTime_Probability"].map("{:.1%}".format)
    top5.columns = ["Airport (City, State)", "Airline",
                    "Delay Probability", "On-Time Probability"]
    return top5

# Cell 10 — User Interaction

Prompts the user for a U.S. state abbreviation, a departure date, and an
optional preferred departure hour. Validates the state against the list of
states present in the dataset, then calls the recommender and prints the
top 5 results with full airline names and airport city labels.

In [10]:
# ===== Cell 10: User Interaction =====

print("=== Flight Delay Predictor (2014-2024 dataset) ===")
print(f"Valid states: {', '.join(valid_states)}\n")

state_input = input("Enter your departure state (e.g. WA): ").strip().upper()
date_input  = input("Enter your departure date (YYYY-MM-DD): ").strip()
hour_raw    = input("Enter preferred departure hour 0-23 (or press Enter for median): ").strip()
hour_input  = int(hour_raw) if hour_raw.isdigit() else None

if state_input not in valid_states:
    print(f"Invalid state '{state_input}'. Please choose from: {', '.join(valid_states)}")
else:
    try:
        results = get_top5_recommendations(state_input, date_input, dep_hour=hour_input)
        if results is not None:
            state_name = STATE_NAMES.get(state_input, state_input)
            hour_str   = f" at hour {hour_input}" if hour_input is not None else ""
            print(f"\nTop 5 recommendations for {state_name} on {date_input}{hour_str}:\n")
            print(results.to_string())
    except Exception as e:
        print(f"Something went wrong: {e}")

=== Flight Delay Predictor (2014-2024 dataset) ===
Valid states: AK, AL, AR, AZ, CA, CO, CT, DE, FL, GA, HI, IA, ID, IL, IN, KS, KY, LA, MA, MD, ME, MI, MN, MO, MS, MT, NC, ND, NE, NH, NJ, NM, NV, NY, OH, OK, OR, PA, RI, SC, SD, TN, TX, UT, VA, VT, WA, WI, WV, WY

Enter your departure state (e.g. WA): WA
Enter your departure date (YYYY-MM-DD): 2026-03-28
Enter preferred departure hour 0-23 (or press Enter for median): 8

Top 5 recommendations for Washington on 2026-03-28 at hour 8:

  Airport (City, State)           Airline Delay Probability On-Time Probability
1           Spokane, WA  Republic Airline             26.5%               73.5%
2           Seattle, WA  Republic Airline             26.6%               73.4%
3           Spokane, WA       Horizon Air             30.6%               69.4%
4            Yakima, WA       Horizon Air             31.2%               68.8%
5        Bellingham, WA       Horizon Air             31.3%               68.7%
